## 텍스트 임베딩하기

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\joyce\AppData\Local\Temp\ipykernel_35348\3828565490.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import pandas as pd
books=pd.read_csv("books_cleaned.csv")

In [4]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing,title_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [5]:
books['tagged_description']

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

일반 text match하는ㄴ것보다 isbn으로 연결하는게 더 깔끔함.

In [7]:
books["tagged_description"].to_csv(
    "tagged_description.txt",
    index=False,
    header=False
)

In [12]:
raw_documents=TextLoader("tagged_description.txt",encoding="utf-8").load()
text_splitter=CharacterTextSplitter(chunk_size=10000,chunk_overlap=0,separator="\n")#separator기준으로 split하도록
documents=text_splitter.split_documents(raw_documents)

In [13]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, G

In [26]:
db_books=Chroma.from_documents(
    documents,
    embedding=OpenAIEmbeddings()
)

In [27]:
query="A book to teach children about nature"
docs=db_books.similarity_search(query,k=10)

In [28]:
docs

[Document(id='17f51181-6859-4bab-974b-010af54a98df', metadata={'source': 'tagged_description.txt'}, page_content='"9780061205699 At the age of eight, Scout Finch is an entrenched free-thinker. She can accept her father\'s warning that it is a sin to kill a mockingbird, because mockingbirds harm no one and give great pleasure. The benefits said to be gained from going to school and keeping her temper elude her. The place of this enchanting, intensely moving story is Maycomb, Alabama. The time is the Depression, but Scout and her brother, Jem, are seldom depressed. They have appalling gifts for entertaining themselves—appalling, that is, to almost everyone except their wise lawyer father, Atticus. Atticus is a man of unfaltering good will and humor, and partly because of this, the children become involved in some disturbing adult mysteries: fascinating Boo Radley, who never leaves his house; the terrible temper of Mrs. Dubose down the street; the fine distinctions that make the Finch fam

지금은 description만 반환-> 우리는 책 정보를 알고싶은데

In [29]:
books[books["isbn13"] == int(
    docs[0].page_content.split()[0].strip('"')
)]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing,title_subtitle,tagged_description
391,9780061205699,0061205699,To Kill a Mockingbird (slipcased edition),Harper Lee,Fiction,http://books.google.com/books/content?id=M9lKH...,"At the age of eight, Scout Finch is an entrenc...",2006.0,4.27,323.0,250.0,0,To Kill a Mockingbird (slipcased edition),"9780061205699 At the age of eight, Scout Finch..."


In [33]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int=10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=50)
    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]
    return books[books["isbn13"].isin(books_list)].head(top_k)

In [34]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing,title_subtitle,tagged_description
114,9780060509057,0060509058,Travels,Michael Crichton,Biography & Autobiography,http://books.google.com/books/content?id=QYilZ...,Often I feel I go to some distant region of th...,2002.0,3.95,400.0,6812.0,0,Travels,9780060509057 Often I feel I go to some distan...
260,9780060852559,0060852550,"Animal, Vegetable, Miracle",Barbara Kingsolver;Camille Kingsolver;Steven L...,Biography & Autobiography,http://books.google.com/books/content?id=qLkEY...,Bestselling author Barbara Kingsolver returns ...,2007.0,4.04,370.0,86130.0,0,"Animal, Vegetable, Miracle: A Year of Food Life",9780060852559 Bestselling author Barbara Kings...
306,9780060932664,006093266X,Collected Novellas,Gabriel Garcia Marquez,Fiction,http://books.google.com/books/content?id=JRcVu...,"Renowned as a master of magical realism, Gabri...",1999.0,4.01,288.0,822.0,0,Collected Novellas,9780060932664 Renowned as a master of magical ...
391,9780061205699,0061205699,To Kill a Mockingbird (slipcased edition),Harper Lee,Fiction,http://books.google.com/books/content?id=M9lKH...,"At the age of eight, Scout Finch is an entrenc...",2006.0,4.27,323.0,250.0,0,To Kill a Mockingbird (slipcased edition),"9780061205699 At the age of eight, Scout Finch..."
442,9780067575208,006757520X,The Sense of Wonder,Rachel Carson,Nature,http://books.google.com/books/content?id=Zee5S...,"First published more than three decades ago, t...",1998.0,4.39,112.0,1160.0,0,The Sense of Wonder,9780067575208 First published more than three ...
690,9780140447897,014044789X,Metamorphosis,Ovid;David Raeburn,Fiction,http://books.google.com/books/content?id=D3McX...,A new translation in hexameter verse of Ovid's...,2004.0,4.05,723.0,46550.0,0,Metamorphosis,9780140447897 A new translation in hexameter v...
880,9780152049676,0152049673,Winter is the Warmest Season,Lauren Stringer,Juvenile Fiction,http://books.google.com/books/content?id=S1EOm...,A child describes pleasant ways to stay warm d...,2006.0,3.91,40.0,360.0,0,Winter is the Warmest Season,9780152049676 A child describes pleasant ways ...
1733,9780375760136,037576013X,Daniel Deronda,George Eliot,Fiction,http://books.google.com/books/content?id=uPiMx...,"Deronda, a high-minded young man searching for...",1876.0,3.83,796.0,19852.0,0,Daniel Deronda,"9780375760136 Deronda, a high-minded young man..."
1911,9780393320039,0393320030,On Becoming a Novelist,John Gardner,Language Arts & Disciplines,http://books.google.com/books/content?id=Go68K...,An indispensable reference for anyone who hope...,1999.0,4.11,150.0,2241.0,0,On Becoming a Novelist,9780393320039 An indispensable reference for a...
2215,9780440420637,0440420636,The Green Toenails Gang,Marjorie Weinman Sharmat,Juvenile Fiction,http://books.google.com/books/content?id=yP1lc...,"Wealthy Olivia Sharp, agent for secrets, uncov...",2005.0,3.74,72.0,41.0,0,The Green Toenails Gang,"9780440420637 Wealthy Olivia Sharp, agent for ..."
